# STL_to_TET

**What this notebook showcases in `gyroid_utils`:** part of the simulation-prep pipeline for a generated gyroid STL. It loads an STL mesh (`io_ops.py`), simplifies it and checks its quality (`mesh_tools.py`), then meshes the resulting surface into a tetrahedral volume mesh using fTetWild and converts it directly to an Abaqus `.inp` file (`TET_mesh_tools.py`).

**Requirements:** a local fTetWild install (`FloatTetwild_bin.exe`). By default this notebook relies on the path hardcoded in `TET_mesh_tools.mesh_an_STL()`; pass `FtetWild_path=...` to that call below if yours is installed elsewhere.


In [ ]:
import os
import numpy as np

import gyroid_utils
from gyroid_utils.utils import reload_all
reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)
parent_dir = os.path.dirname(working_path)


In [ ]:
#==========================================================================================
#================================ load your STL file ======================================
#==========================================================================================
file_name = "gyroid-constant-1"
verts, faces = gyroid_utils.io_ops.load_stl(parent_dir + '/WP1 - stl files/' + file_name + '.stl')

# ------ visualize the quality of your mesh ----
triangle_areas = gyroid_utils.mesh_tools.calculate_triangle_areas(verts, faces)
gyroid_utils.viz.view_mesh(faces, verts)
gyroid_utils.viz.plot_histogram(triangle_areas)
gyroid_utils.mesh_tools.check_mesh_validity(verts, faces)


In [ ]:
#==========================================================================================
#================================ simplify the mesh =======================================
#==========================================================================================
verts, faces = gyroid_utils.mesh_tools.simplify_mesh(verts, faces, target=400000)
gyroid_utils.viz.view_mesh(faces, verts)
gyroid_utils.mesh_tools.check_mesh_validity(verts, faces)


In [ ]:
#==========================================================================================
#===================== export the simplified mesh to a new STL ============================
#==========================================================================================
# TET_mesh_tools.mesh_an_STL() (below) meshes an STL file that lives on disk with
# fTetWild, so the simplified in-memory mesh needs to be written out first.
tet_file_name = f"{file_name}_simplified"
simplified_stl_path = os.path.join(working_path, tet_file_name + ".stl")
gyroid_utils.mesh_tools.export_as_STL(verts, faces, simplified_stl_path)
print(f"Simplified STL exported: {simplified_stl_path}")


In [ ]:
#==========================================================================================
#============== mesh with fTetWild and convert to an Abaqus .inp file =====================
#==========================================================================================
# input_path/output_path are folder paths and must end with a path separator,
# since mesh_an_STL() builds file paths via plain string concatenation.
input_path = working_path + os.sep
output_path = working_path + os.sep

gyroid_utils.TET_mesh_tools.mesh_an_STL(
    input_path=input_path,
    output_path=output_path,
    file_name=tet_file_name,
    stop_energy=20.0,
    epsilon=0.001,
    CPU_cores=1,
    print_outputs=True,
    # FtetWild_path="C:\\Program Files\\fTetWild\\build\\Release\\FloatTetwild_bin.exe",  # uncomment/edit if fTetWild is installed elsewhere
)

inp_path = os.path.join(output_path, tet_file_name + ".inp")
print(f"Abaqus mesh saved: {inp_path}")


In [ ]:
#==========================================================================================
#================================ mesh summary =============================================
#==========================================================================================
num_nodes, num_elements = gyroid_utils.TET_mesh_tools.get_mesh_info(inp_path)
print(f"Tetrahedral mesh: {num_nodes} nodes, {num_elements} elements")
